# Hands-on Modul 2.6: Ekstraksi Data Terstruktur dengan Pydantic & Instructor 🛠️

Tantangan terbesar LLM di dunia nyata bukan "membuat puisi", tapi **mengintegrasikannya ke sistem lain**. Database butuh struktur (JSON/SQL), bukan teks narasi.

Di hands-on ini, kita akan membangun *pipeline* ekstraksi yang **tahan banting** (robust). Kita tidak hanya meminta JSON, tapi kita **memaksa** struktur dan tipe data yang benar menggunakan Pydantic.

In [1]:
# Instalasi Library
# 'pydantic': Standar validasi data Python
# 'openai': Klien API standar
# 'instructor': Library magic untuk structured output
!pip install pydantic openai instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.8/358.8 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.13.0
    Uninstalling jiter-0.13.0:
      Successfully uninstalled jiter-0.13.0


In [ ]:
import os
import instructor
from openai import OpenAI

# --- SETUP API KEY ---
# Masukkan API Key OpenAI Anda di sini
# Jika tidak punya, biarkan kosong (nanti kita pakai mode simulasi/mock di bawah)
os.environ["OPENAI_API_KEY"] = ""

# Cek apakah key ada
has_key = os.environ.get("OPENAI_API_KEY") and os.environ.get("OPENAI_API_KEY").startswith("sk-")

if has_key:
    print("✅ API Key terdeteksi. Menggunakan mode Live.")
    # Patching Client OpenAI dengan Instructor
    client = instructor.from_openai(OpenAI())
else:
    print("⚠️ API Key tidak valid/kosong. Masuk ke Mode Mock (Simulasi).")
    # (Kode simulasi ada di sel berikutnya agar tidak error)

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# 1. Definisikan "Bentuk Data" yang kita inginkan
# Bayangkan kita ingin mengekstrak data dari struk belanja
class ReceiptItem(BaseModel):
    product_name: str = Field(..., description="Nama produk yang dibeli")
    quantity: int = Field(..., description="Jumlah barang (harus integer)")
    price_per_unit: float = Field(..., description="Harga satuan")
    total_price: float = Field(..., description="Total harga baris ini")

class ReceiptExtraction(BaseModel):
    store_name: str = Field(..., description="Nama toko")
    date: str = Field(..., description="Tanggal transaksi dalam format YYYY-MM-DD")
    items: List[ReceiptItem] = Field(..., description="Daftar item belanjaan")
    grand_total: float = Field(..., description="Total belanjaan akhir")
    is_suspicious: bool = Field(
        False,
        description="True jika ada item yang harganya tidak masuk akal (misal > 10 juta untuk makanan)"
    )

# Lihat Skema JSON yang dihasilkan otomatis
import json
print("--- Auto-Generated JSON Schema ---")
print(json.dumps(ReceiptExtraction.model_json_schema(), indent=2))

--- Auto-Generated JSON Schema ---
{
  "$defs": {
    "ReceiptItem": {
      "properties": {
        "product_name": {
          "description": "Nama produk yang dibeli",
          "title": "Product Name",
          "type": "string"
        },
        "quantity": {
          "description": "Jumlah barang (harus integer)",
          "title": "Quantity",
          "type": "integer"
        },
        "price_per_unit": {
          "description": "Harga satuan",
          "title": "Price Per Unit",
          "type": "number"
        },
        "total_price": {
          "description": "Total harga baris ini",
          "title": "Total Price",
          "type": "number"
        }
      },
      "required": [
        "product_name",
        "quantity",
        "price_per_unit",
        "total_price"
      ],
      "title": "ReceiptItem",
      "type": "object"
    }
  },
  "properties": {
    "store_name": {
      "description": "Nama toko",
      "title": "Store Name",
      "type": "string

In [ ]:
# Teks Input (Unstructured Data)
# Ini contoh teks OCR yang berantakan dari struk
receipt_text = """
MINIMARKET MAJU JAYA
Tgl: 12-Nov-2024
----------------------
Roti Tawar    2x   15.000   30.000
Susu UHT      1x   20.000   20.000
Permen Karet  5x    1.000    5.000
----------------------
Total: 55.000
"""

print(f"Input Teks:\n{receipt_text}\n")

if has_key:
    # --- MODE LIVE (Panggil LLM) ---
    try:
        extraction = client.chat.completions.create(
            model="gpt-3.5-turbo",
            response_model=ReceiptExtraction, # <--- MAGIC: Langsung minta Class Pydantic
            max_retries=3, # Auto-fix jika validasi gagal
            messages=[
                {"role": "user", "content": f"Ekstrak data struk ini: {receipt_text}"}
            ]
        )
        print("✅ Ekstraksi Berhasil!")
        print("-" * 30)
        print(f"Toko: {extraction.store_name}")
        print(f"Tanggal (Normalized): {extraction.date}") # Perhatikan format tanggalnya jadi YYYY-MM-DD!
        print(f"Total Item: {len(extraction.items)}")
        print(f"Data Valid? {isinstance(extraction, ReceiptExtraction)}") # True

        # Akses data item pertama
        item1 = extraction.items[0]
        print(f"Item 1: {item1.product_name} (Qty: {item1.quantity})")

    except Exception as e:
        print(f"Error: {e}")

else:
    # --- MODE MOCK (Simulasi Manual) ---
    print("ℹ️ Menjalankan Simulasi Validasi Pydantic (Tanpa LLM)...")

    # Simulasi output LLM (JSON mentah)
    mock_llm_output = {
        "store_name": "MINIMARKET MAJU JAYA",
        "date": "2024-11-12", # LLM cerdas mengubah '12-Nov-2024' jadi ISO format
        "items": [
            {"product_name": "Roti Tawar", "quantity": 2, "price_per_unit": 15000, "total_price": 30000},
            {"product_name": "Susu UHT", "quantity": 1, "price_per_unit": 20000, "total_price": 20000},
             # Simulasi typo string "5" jadi int 5 (Type Coercion)
            {"product_name": "Permen Karet", "quantity": "5", "price_per_unit": 1000, "total_price": 5000}
        ],
        "grand_total": 55000,
        "is_suspicious": False
    }

    # Validasi Manual
    try:
        extraction = ReceiptExtraction(**mock_llm_output)
        print("✅ Validasi Pydantic Berhasil!")
        print(f"Toko: {extraction.store_name}")
        print(f"Item 3 Qty (Tipe): {type(extraction.items[2].quantity)}") # <class 'int'>
        print("Perhatikan: String '5' otomatis diubah jadi Integer 5 oleh Pydantic!")
    except Exception as e:
        print(f"Validasi Gagal: {e}")

Input Teks:

MINIMARKET MAJU JAYA
Tgl: 12-Nov-2024
----------------------
Roti Tawar    2x   15.000   30.000
Susu UHT      1x   20.000   20.000
Permen Karet  5x    1.000    5.000
----------------------
Total: 55.000


ℹ️ Menjalankan Simulasi Validasi Pydantic (Tanpa LLM)...
✅ Validasi Pydantic Berhasil!
Toko: MINIMARKET MAJU JAYA
Item 3 Qty (Tipe): <class 'int'>
Perhatikan: String '5' otomatis diubah jadi Integer 5 oleh Pydantic!


### 💡 Analisis Self-Correction (Teori)

Jika Anda menggunakan Mode Live, perhatikan bahwa `max_retries=3` diaktifkan.
Apa yang terjadi jika LLM salah (misal: `grand_total` diisi string "lima puluh ribu")?

1.  **Pydantic Error:** `ValidationError: value is not a valid float`
2.  **Instructor Catch:** Library menangkap error ini.
3.  **Retry:** Instructor mengirim pesan baru ke LLM: *"Error in field 'grand_total': value is not a valid float. Please fix."*
4.  **LLM Correction:** LLM mengenerate ulang JSON dengan `grand_total: 50000`.
5.  **Success:** Data valid dikembalikan ke Anda.

Inilah yang membuat *pipeline* ini **robust** untuk produksi.